# 🇳🇵 Nepal SSF Pension Calculator

**Based on Contribution-Based Social Security Act, 2074 (2017)**

---

### ▶️ How to run
Click **Runtime → Run all** from the menu above.

The calculator will ask you 4 questions. Just type your answers and press **Enter** each time.

---


In [ ]:
# ══════════════════════════════════════════════════════════════
#   Nepal SSF Pension Calculator
#   Based on SSF Act 2074 / Scheme Operation Procedure 2075
# ══════════════════════════════════════════════════════════════

def format_npr(amount):
    amount = round(amount)
    sign = "-" if amount < 0 else ""
    s = str(abs(amount))
    if len(s) <= 3:
        return f"NPR {sign}{s}"
    last3 = s[-3:]
    rest = s[:-3]
    parts = []
    while len(rest) > 2:
        parts.append(rest[-2:])
        rest = rest[:-2]
    if rest:
        parts.append(rest)
    parts.reverse()
    return f"NPR {sign}{','.join(parts)},{last3}"


def calculate_ssf(basic_salary, current_age, annual_increment_pct=2.0,
                  annual_return_pct=0.0, salary_cap=100_000, retirement_age=60):
    if current_age >= retirement_age:
        raise ValueError(f"Current age ({current_age}) must be less than {retirement_age}.")

    years_to_retire = retirement_age - current_age
    PENSION_RATE   = 0.20
    GRATUITY_RATE  = 0.0833
    INSURANCE_RATE = 0.0267
    EMPLOYEE_RATE  = 0.11
    EMPLOYER_RATE  = 0.20

    monthly_return = (1 + annual_return_pct / 100.0) ** (1 / 12) - 1
    pension_corpus = gratuity_corpus = 0.0
    total_employee_paid = total_employer_paid = 0.0
    total_pension_contrib = total_gratuity_contrib = total_insurance_consumed = 0.0

    salary = basic_salary
    yearly_breakdown = []

    for year in range(1, years_to_retire + 1):
        yr_pension = yr_gratuity = yr_insurance = yr_employee = yr_employer = 0.0
        for _ in range(12):
            s = min(salary, salary_cap)
            pension_corpus  = pension_corpus  * (1 + monthly_return) + s * PENSION_RATE
            gratuity_corpus = gratuity_corpus * (1 + monthly_return) + s * GRATUITY_RATE
            yr_pension   += s * PENSION_RATE
            yr_gratuity  += s * GRATUITY_RATE
            yr_insurance += s * INSURANCE_RATE
            yr_employee  += s * EMPLOYEE_RATE
            yr_employer  += s * EMPLOYER_RATE
        total_pension_contrib    += yr_pension
        total_gratuity_contrib   += yr_gratuity
        total_insurance_consumed += yr_insurance
        total_employee_paid      += yr_employee
        total_employer_paid      += yr_employer
        yearly_breakdown.append({
            "year": year, "age": current_age + year,
            "monthly_basic_effective": round(min(salary, salary_cap), 2),
            "pension_corpus_eoy":  round(pension_corpus, 2),
            "gratuity_corpus_eoy": round(gratuity_corpus, 2),
        })
        salary *= (1 + annual_increment_pct / 100.0)

    monthly_pension = pension_corpus / 160
    return {
        "inputs": {
            "basic_salary": basic_salary,
            "effective_start_salary": min(basic_salary, salary_cap),
            "current_age": current_age, "retirement_age": retirement_age,
            "years_to_retire": years_to_retire,
            "annual_increment_pct": annual_increment_pct,
            "annual_return_pct": annual_return_pct, "salary_cap": salary_cap,
        },
        "totals": {
            "total_employee_paid":      round(total_employee_paid, 2),
            "total_employer_paid":      round(total_employer_paid, 2),
            "total_all_paid":           round(total_employee_paid + total_employer_paid, 2),
            "total_pension_contrib":    round(total_pension_contrib, 2),
            "total_gratuity_contrib":   round(total_gratuity_contrib, 2),
            "total_insurance_consumed": round(total_insurance_consumed, 2),
        },
        "pension_corpus":       round(pension_corpus, 2),
        "gratuity_corpus":      round(gratuity_corpus, 2),
        "total_old_age_corpus": round(pension_corpus + gratuity_corpus, 2),
        "monthly_pension":      round(monthly_pension, 2),
        "annual_pension":       round(monthly_pension * 12, 2),
        "yearly_breakdown":     yearly_breakdown,
    }


def print_report(result, label=""):
    inp = result["inputs"]
    tot = result["totals"]
    eff = inp["effective_start_salary"]
    W = 65
    header = "NEPAL SSF PENSION CALCULATOR"
    if label:
        header += f"  [{label}]"
    print("\n" + "=" * W)
    print(f"  {header}")
    print("=" * W)

    print("\n📥  YOUR INPUTS")
    print(f"   Monthly basic salary         : {format_npr(inp['basic_salary'])}")
    if inp["basic_salary"] > inp["salary_cap"]:
        print(f"   ⚠  Salary capped at          : {format_npr(inp['salary_cap'])}")
    print(f"   Current age                  : {inp['current_age']} yrs")
    print(f"   Retirement age               : {inp['retirement_age']} yrs")
    print(f"   Years until retirement       : {inp['years_to_retire']} yrs")
    print(f"   Annual salary increment      : {inp['annual_increment_pct']}%")
    print(f"   Annual return on corpus      : {inp['annual_return_pct']}%")

    print("\n📊  MONTHLY CONTRIBUTION BREAKDOWN  (at starting salary)")
    print(f"   {'─'*55}")
    print(f"   EMPLOYEE pays 11%            : {format_npr(eff * 0.11)}")
    print(f"     → Pension Fund (10%)       : {format_npr(eff * 0.10)}")
    print(f"     → Medical insurance (1%)   : {format_npr(eff * 0.01)}  ❌ not withdrawable")
    print(f"   {'─'*55}")
    print(f"   EMPLOYER pays 20%            : {format_npr(eff * 0.20)}")
    print(f"     → Pension Fund (10%)       : {format_npr(eff * 0.10)}")
    print(f"     → Gratuity Fund (8.33%)    : {format_npr(eff * 0.0833)}")
    print(f"     → Accident ins. (1.4%)     : {format_npr(eff * 0.014)}  ❌ not withdrawable")
    print(f"     → Dependent ins. (0.27%)   : {format_npr(eff * 0.0027)}  ❌ not withdrawable")
    print(f"   {'─'*55}")
    print(f"   TOTAL (31%)                  : {format_npr(eff * 0.31)}")
    print(f"     → Pension Fund (20%)       : {format_npr(eff * 0.20)}  🔒 locked until 60")
    print(f"     → Gratuity Fund (8.33%)    : {format_npr(eff * 0.0833)}  ✅ withdrawable anytime")
    print(f"     → Insurance total (2.67%)  : {format_npr(eff * 0.0267)}  ❌ consumed as coverage")

    print("\n📈  LIFETIME CONTRIBUTION TOTALS")
    print(f"   Total paid by employee       : {format_npr(tot['total_employee_paid'])}")
    print(f"   Total paid by employer       : {format_npr(tot['total_employer_paid'])}")
    print(f"   {'─'*55}")
    print(f"   Total paid into SSF          : {format_npr(tot['total_all_paid'])}")
    print(f"     → Into Pension Fund (20%)  : {format_npr(tot['total_pension_contrib'])}")
    print(f"     → Into Gratuity Fund(8.33%): {format_npr(tot['total_gratuity_contrib'])}")
    print(f"     → Insurance consumed(2.67%): {format_npr(tot['total_insurance_consumed'])}  ❌ gone")

    print("\n🏦  ACCUMULATED CORPUS AT RETIREMENT (age 60)")
    ret_note = f"+ {inp['annual_return_pct']}% p.a. returns" if inp["annual_return_pct"] > 0 else "no investment returns"
    print(f"   [{ret_note}]")
    print(f"   Pension corpus  (20%)        : {format_npr(result['pension_corpus'])}")
    print(f"   Gratuity corpus (8.33%)      : {format_npr(result['gratuity_corpus'])}")
    print(f"   Total Old Age corpus (28.33%): {format_npr(result['total_old_age_corpus'])}")

    print("\n💼  GRATUITY  (lump sum — available on job termination)")
    print(f"   ┌{'─'*55}┐")
    print(f"   │  The 8.33% gratuity corpus can be withdrawn as a       │")
    print(f"   │  lump sum ANYTIME you leave your job (before or at 60).│")
    print(f"   │                                                         │")
    print(f"   │  Gratuity amount : {format_npr(result['gratuity_corpus']):<36} │")
    print(f"   │                                                         │")
    print(f"   │  ⚠  The 20% pension corpus CANNOT be withdrawn early.  │")
    print(f"   │     It is locked and paid only as monthly pension.      │")
    print(f"   └{'─'*55}┘")

    print("\n🏖️  MONTHLY PENSION  (from age 60, if 15+ years contributed)")
    print(f"   ┌{'─'*55}┐")
    print(f"   │  Formula: Pension Corpus (20%) ÷ 160                    │")
    print(f"   │                                                         │")
    print(f"   │  Pension corpus  : {format_npr(result['pension_corpus']):<36} │")
    print(f"   │  Monthly pension : {format_npr(result['monthly_pension']):<36} │")
    print(f"   │  Annual pension  : {format_npr(result['annual_pension']):<36} │")
    print(f"   └{'─'*55}┘")

    if inp["years_to_retire"] < 15:
        print(f"\n   ⚠  WARNING: Only {inp['years_to_retire']} yrs of contributions.")
        print(f"      Monthly pension requires 15+ years. Gratuity still available.")

    print("\n📅  YEAR-BY-YEAR GROWTH  (every 5 years + first + last)")
    print(f"   {'Yr':<5} {'Age':<5} {'Basic (eff)':<16} {'Pension Corpus':<18} {'Gratuity Corpus'}")
    print(f"   {'─'*68}")
    for row in result["yearly_breakdown"]:
        if row["year"] == 1 or row["year"] % 5 == 0 or row["year"] == inp["years_to_retire"]:
            print(
                f"   {row['year']:<5} {row['age']:<5} "
                f"{format_npr(row['monthly_basic_effective']):<16} "
                f"{format_npr(row['pension_corpus_eoy']):<18} "
                f"{format_npr(row['gratuity_corpus_eoy'])}"
            )

    print("\n" + "=" * W)
    print("  ⓘ  Figures are projections. Actual SSF returns set by SSF Board.")
    print("  ⓘ  Pension reviewed every 3 years for inflation.")
    print("  ⓘ  Source: SSF Act 2074 / Scheme Operation Procedure 2075")
    print("=" * W + "\n")


def ask_float(prompt, default=None):
    while True:
        suffix = f" [Enter = {default}]" if default is not None else ""
        raw = input(f"{prompt}{suffix}: ").strip()
        if raw == "" and default is not None:
            return default
        try:
            val = float(raw)
            if val < 0:
                print("  ⚠  Please enter a positive number.")
                continue
            return val
        except ValueError:
            print("  ⚠  Please enter a valid number (e.g. 50000).")

def ask_int(prompt, min_val=0, max_val=120):
    while True:
        raw = input(f"{prompt}: ").strip()
        try:
            val = int(raw)
            if val < min_val or val > max_val:
                print(f"  ⚠  Please enter a value between {min_val} and {max_val}.")
                continue
            return val
        except ValueError:
            print("  ⚠  Please enter a whole number (e.g. 30).")


# ── MAIN ──────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════╗")
print("║       Nepal SSF Pension Calculator               ║")
print("║   Based on SSF Act 2074 & Scheme Rules 2075      ║")
print("╚══════════════════════════════════════════════════╝")
print()
print("Please answer the 4 questions below.")
print("Press Enter to accept the default value shown in [ ].")
print("─" * 52)
print()

basic_salary = ask_float("  1. Monthly BASIC salary (NPR)")
current_age  = ask_int(  "  2. Your current age (18–59)", min_val=18, max_val=59)
increment    = ask_float("  3. Annual salary increment %", default=2.0)
return_rate  = ask_float("  4. Assumed annual return on SSF corpus %", default=0.0)

print()
print("  ✅  Calculating your SSF projection...")
print()

try:
    label = f"{return_rate}% return" if return_rate > 0 else "0% return — conservative"
    result_base = calculate_ssf(basic_salary, current_age, increment, return_rate)
    print_report(result_base, label=label)

    result_5 = calculate_ssf(basic_salary, current_age, increment, 5.0)

    print("─" * 65)
    print("  📊  COMPARISON: your scenario vs 5% annual return")
    print("─" * 65)
    base_label = f"{return_rate}% return" if return_rate != 0.0 else "0% (no returns)"
    print(f"  {'Metric':<38} {base_label:<20} {'5% annual return'}")
    print(f"  {'─'*75}")
    for lbl, key in [
        ("Pension corpus (20%)",    "pension_corpus"),
        ("Gratuity corpus (8.33%)", "gratuity_corpus"),
        ("Total old age corpus",    "total_old_age_corpus"),
        ("Monthly pension (÷160)",  "monthly_pension"),
        ("Annual pension",          "annual_pension"),
    ]:
        print(f"  {lbl:<38} {format_npr(result_base[key]):<20} {format_npr(result_5[key])}")
    print()

except ValueError as e:
    print(f"\n  ❌ Error: {e}\n")


---
### 📌 SSF Contribution Rules at a Glance

| Portion | Rate | What happens |
|---|---|---|
| **Pension Fund** | 20% | 🔒 Locked until age 60 — paid as monthly pension (corpus ÷ 160) |
| **Gratuity Fund** | 8.33% | ✅ Withdrawable as lump sum when you leave your job |
| **Medical / Maternity** | 1.00% | ❌ Consumed — covers up to NPR 1,00,000/yr at empanelled hospitals |
| **Accident & Disability** | 1.40% | ❌ Consumed — workplace accident coverage from day 1 |
| **Dependent Family** | 0.27% | ❌ Consumed — pension for spouse/children if you pass away |
| **Total** | **31%** | |

> Source: [ssf.gov.np](https://ssf.gov.np) · SSF Act 2074 · Scheme Operation Procedure 2075
